In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Add project root to path
sys.path.append(str(Path("..").resolve()))

# Laden von trainierten Modellen


In [2]:
from helpers import load_fitted_model

model_configurations = [
    ("hmm", 2, 42),
    ("hmm", 3, 42),
    ("hmm", 4, 42),  # Konfigurationen mit Seed 42
    ("vdhmm", 2, 42),
    ("vdhmm", 3, 42),
    ("vdhmm", 4, 42),
    ("hmm", 2, 43),
    ("hmm", 3, 43),
    ("hmm", 4, 43),
    ("vdhmm", 2, 43),
    ("vdhmm", 3, 43),  # Konfigurationen mit Seed 43
    ("vdhmm", 4, 43),
    ("hmm", 2, 123),
    ("hmm", 3, 123),
    ("hmm", 4, 123),  # Konfigrationen mit Seed 123
    ("vdhmm", 2, 123),
    ("vdhmm", 3, 123),
    ("vdhmm", 4, 123),
    ("hmm", 2, None),  # Konfiguration mit Seed None (Original Indizes)
    ("hmm", 3, None),
    ("hmm", 4, None),
    ("vdhmm", 2, None),  # Konfiguration mit Seed None (Original Indizes)
    ("vdhmm", 3, None),
    ("vdhmm", 4, None),
]

n_models = len(model_configurations)

# get different metrics out of hmm models

results_df = pd.DataFrame(
    {
        "Model": [
            f"{model_name}_{S}" for (model_name, S, seed) in model_configurations
        ],
        "Seed": [
            seed if seed else "Original"
            for (model_name, S, seed) in model_configurations
        ],
        # "LOOIC": [0.0] * n_models,
        # "WAIC": [0.0] * n_models,
        # "LPD_TRAIN": [0.0] * n_models,
        # "LPD_VAL": [0.0] * n_models,
        "AUC_TRAIN": [0.0] * n_models,
        "AUC_TEST": [0.0] * n_models,
        "R_HAT_MEAN": [0.0] * n_models,
        "R_HAT_STD": [0.0] * n_models,
    }
)

/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import arviz as az
from sklearn.metrics import roc_auc_score
import warnings

from constants import PROCESSED_DATA_FOLDER
from helpers import ModelData

# Unterdrücke ArviZ Warnungen und Output
warnings.filterwarnings("ignore")

# Schleife über alle Modell-Konfigurationen
for k, (model_name, S, seed) in enumerate(model_configurations):
    print(f"Processing {k+1}/{n_models}: {model_name} with {S} states (seed={seed})...")
    model_data = ModelData.from_pickle(
        PROCESSED_DATA_FOLDER / f"processed_data_{seed if seed else "original"}.pkl"
    )
    eval_indices = model_data.eval_indices

    # get original eval indices for each seed
    np.random.seed(seed)
    indices_val_cal = np.random.permutation(np.arange(500, 921))

    # Lade das Modell mit Seed
    try:
        model = load_fitted_model(model_name, S, seed=seed)
    except FileNotFoundError:
        print(
            f"FileNotFound: {model_name} with {S} states and seed {seed} was not found! Skipping..."
        )
        continue

    log_lik = model.fit.stan_variable("log_lik")
    log_lik_test = model.fit.stan_variable("log_lik_test")
    close_prob = model.fit.stan_variable("close_prob")
    close_prob_mean = close_prob.mean(axis=0)

    idata = az.from_cmdstanpy(model.fit, log_likelihood="log_lik")

    # Berechne looc, waic, lpd (ohne Output)
    # results_df.loc[k, "LOOIC"] = az.loo(idata, var_name="log_lik").elpd_loo
    # results_df.loc[k, "WAIC"] = az.waic(idata, var_name="log_lik").elpd_waic
    # results_df.loc[k, "LPD_TRAIN"] = -2 * np.sum(np.log(np.exp(log_lik).mean(axis=0)))
    # results_df.loc[k, "LPD_VAL"] = -2 * np.sum(
    #     np.log(np.exp(log_lik_test).mean(axis=0))
    # )

    # Berechne AUC für Trainingsdaten (in Prozent)
    y_true_train = np.array(model.stan_data["Closed"][:500])
    results_df.loc[k, "AUC_TRAIN"] = 100 * roc_auc_score(
        y_true_train, close_prob_mean[:500]
    )

    y_true_test = np.array(model.stan_data["Closed"])[eval_indices]
    # Annahme: Die Schrittweite passt, ggf. muss die Dimensionalität geprüft werden!
    close_prob_test_mean = np.mean(model.fit.stan_variable("close_prob"), axis=0)
    results_df.loc[k, "AUC_TEST"] = 100 * roc_auc_score(
        y_true_test, close_prob_mean[eval_indices]
    )

    # Verwende das bereits gespeicherte summary DataFrame (bereits in model.summary!)
    # Berechne Durchschnitt und Standardabweichung von R_hat über alle Parameter
    results_df.loc[k, "R_HAT_MEAN"] = np.nanmean(model.summary["R_hat"])
    results_df.loc[k, "R_HAT_STD"] = np.nanstd(model.summary["R_hat"])

print("\nErgebnisse:")

# Ergebnisse runden
results_df["AUC_TRAIN"] = results_df["AUC_TRAIN"].round(2)
results_df["AUC_TEST"] = results_df["AUC_TEST"].round(2)
results_df["R_HAT_MEAN"] = results_df["R_HAT_MEAN"].round(4)
results_df["R_HAT_STD"] = results_df["R_HAT_STD"].round(4)


np.random.seed(None)

results_df

Processing 1/24: hmm with 2 states (seed=42)...
Processing 2/24: hmm with 3 states (seed=42)...
Processing 3/24: hmm with 4 states (seed=42)...
Processing 4/24: vdhmm with 2 states (seed=42)...
Processing 5/24: vdhmm with 3 states (seed=42)...
Processing 6/24: vdhmm with 4 states (seed=42)...
Processing 7/24: hmm with 2 states (seed=43)...
Processing 8/24: hmm with 3 states (seed=43)...
Processing 9/24: hmm with 4 states (seed=43)...
Processing 10/24: vdhmm with 2 states (seed=43)...
Processing 11/24: vdhmm with 3 states (seed=43)...
Processing 12/24: vdhmm with 4 states (seed=43)...
Processing 13/24: hmm with 2 states (seed=123)...
Processing 14/24: hmm with 3 states (seed=123)...
Processing 15/24: hmm with 4 states (seed=123)...
FileNotFound: hmm with 4 states and seed 123 was not found! Skipping...
Processing 16/24: vdhmm with 2 states (seed=123)...
Processing 17/24: vdhmm with 3 states (seed=123)...
Processing 18/24: vdhmm with 4 states (seed=123)...
Processing 19/24: hmm with 2 st

,Model,Seed,AUC_TRAIN,AUC_TEST,R_HAT_MEAN,R_HAT_STD
0,hmm_2,42,79.86,78.44,1.0006,0.0007
1,hmm_3,42,83.83,81.34,1.3394,0.2104
2,hmm_4,42,83.60,80.78,1.6289,0.4337
3,vdhmm_2,42,79.26,78.32,1.3574,0.2292
4,vdhmm_3,42,85.36,83.27,1.0010,0.0009
5,vdhmm_4,42,85.78,83.72,1.5874,0.3723
6,hmm_2,43,82.24,73.80,1.0006,0.0007
7,hmm_3,43,84.48,79.44,1.3344,0.2133
8,hmm_4,43,85.90,80.74,1.2905,0.2101
9,vdhmm_2,43,82.85,72.32,1.0008,0.0008


In [6]:
grouped_dfs = {}

for seed, group_df in results_df.groupby("Seed"):
    grouped_dfs[seed] = group_df.copy()

print("SEED 42:")
display(grouped_dfs[42])

print("SEED 43:")
display(grouped_dfs[43])

print("SEED 123:")
display(grouped_dfs[123])

print("ORIGINAL INDICES")
display(grouped_dfs["Original"])

SEED 42:


,Model,Seed,AUC_TRAIN,AUC_TEST,R_HAT_MEAN,R_HAT_STD
0,hmm_2,42,79.86,78.44,1.0006,0.0007
1,hmm_3,42,83.83,81.34,1.3394,0.2104
2,hmm_4,42,83.60,80.78,1.6289,0.4337
3,vdhmm_2,42,79.26,78.32,1.3574,0.2292
4,vdhmm_3,42,85.36,83.27,1.0010,0.0009
5,vdhmm_4,42,85.78,83.72,1.5874,0.3723


SEED 43:


,Model,Seed,AUC_TRAIN,AUC_TEST,R_HAT_MEAN,R_HAT_STD
6,hmm_2,43,82.24,73.80,1.0006,0.0007
7,hmm_3,43,84.48,79.44,1.3344,0.2133
8,hmm_4,43,85.90,80.74,1.2905,0.2101
9,vdhmm_2,43,82.85,72.32,1.0008,0.0008
10,vdhmm_3,43,86.14,83.52,1.0019,0.0017
11,vdhmm_4,43,86.32,81.18,1.7999,0.5237


SEED 123:


,Model,Seed,AUC_TRAIN,AUC_TEST,R_HAT_MEAN,R_HAT_STD
12,hmm_2,123,78.44,75.11,1.3746,0.2151
13,hmm_3,123,82.20,77.15,1.4801,0.2884
14,hmm_4,123,0.00,0.00,0.0000,0.0000
15,vdhmm_2,123,78.21,74.93,1.0009,0.0008
16,vdhmm_3,123,84.19,79.99,1.7911,0.4310
17,vdhmm_4,123,84.19,80.52,1.4819,0.3928


ORIGINAL INDICES


,Model,Seed,AUC_TRAIN,AUC_TEST,R_HAT_MEAN,R_HAT_STD
18,hmm_2,Original,78.53,77.14,1.0010,0.0011
19,hmm_3,Original,81.81,82.39,1.3345,0.2149
20,hmm_4,Original,82.68,83.71,1.6543,0.4159
21,vdhmm_2,Original,78.27,76.40,1.3539,0.2286
22,vdhmm_3,Original,83.93,84.39,1.0007,0.0008
23,vdhmm_4,Original,82.99,83.48,1.6784,0.4346
